# GitHub Customer Churn Predictor - EDA and Feature Selection

This notebook documents the exploratory data analysis, feature engineering, churn label distribution, and feature selection process used in the GitHub customer churn prediction project.

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
from sklearn.feature_selection import VarianceThreshold
from app.features import generate_features, FEATURE_COLUMNS
from app.model import run_feature_selection

pd.set_option('display.max_columns', None)

## 2. Load Raw Dataset

The raw dataset was collected from the GitHub REST API and stored as a CSV file.

In [ ]:
raw_df = pd.read_csv('../data/raw/github_users.csv')
raw_df.head()

In [ ]:
raw_df.shape

## 3. Generate Features

The raw GitHub profile attributes are transformed into numerical features used by the machine learning model.

In [ ]:
df = generate_features(raw_df)
df[FEATURE_COLUMNS + ['churned']].head()

## 4. Churn Label Distribution

Churn is defined as a user having more than 90 days since last activity.

In [ ]:
df['churned'].value_counts()

In [ ]:
df['churned'].value_counts(normalize=True).round(3)

## 5. Feature Summary

In [ ]:
df[FEATURE_COLUMNS].describe().T

## 6. Filter Method - Variance Threshold

Variance threshold is used to detect features with no variation or very low variation. Features with zero variance do not help the model because they contain the same value for every record.

In [ ]:
X = df[FEATURE_COLUMNS]

variances = pd.DataFrame({
    'feature': FEATURE_COLUMNS,
    'variance': X.var().values
}).sort_values('variance')

variances

In [ ]:
constant_features = variances[variances['variance'] == 0]
constant_features

## 7. Filter Method - Correlation Matrix

The correlation matrix helps detect relationships between features and possible redundancy among variables.

In [ ]:
corr = df[FEATURE_COLUMNS + ['churned']].corr(numeric_only=True)
corr.round(3)

In [ ]:
corr['churned'].sort_values(ascending=False).round(3)

## 8. Feature Selection Comparison

The project compares four feature selection approaches:

1. Filter method using ANOVA F-test
2. Recursive Feature Elimination using Logistic Regression
3. Decision Tree feature importance
4. Random Forest feature importance

In [ ]:
selection = run_feature_selection(df)
selection

In [ ]:
selection[selection['decision'] == 'Keep']

## 9. Model Comparison

A second model was trained without `days_since_last_activity` to evaluate whether the model was relying too strongly on the feature used to define churn.

In [ ]:
model_comparison = pd.read_csv('../data/raw/model_comparison.csv')
model_comparison

## 10. Conclusion

The strongest feature is `days_since_last_activity`, which is expected because the churn label is based on inactivity. To avoid relying only on that direct signal, a second model was evaluated without the recency feature. The lower but more realistic performance of the second model shows that indirect engagement and profile-based signals still provide predictive value.

This notebook supports the complete workflow required for the project: data loading, feature generation, churn distribution analysis, variance threshold, correlation matrix, feature selection comparison, and model evaluation.